In [12]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型（老師提供，請勿修改）
# ============================================================
!pip install -q groq gradio

import os
import re
import time
import sqlite3
import pandas as pd
from IPython.display import display
from groq import Groq
import gradio as gr

# 判斷目前是在 Colab 還是本地端 Jupyter
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 🔴 金鑰名稱一律為 GROQ_API_KEY；絕對不可以把金鑰寫在程式裡，也不可以 print 出來
if IN_COLAB:
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        print("❌ 讀不到金鑰：請到左側 🔑 新增名稱為 GROQ_API_KEY 的密鑰，並打開「筆記本存取權」")
        raise
else:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise RuntimeError("找不到環境變數 GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

# 期中考指定模型；推理強度設為 low，可大幅減少 token 用量
# （唯一允許修改處：若 20b 額度用完，可改為 "openai/gpt-oss-120b"，並在規格書最後註明）
MODEL = "openai/gpt-oss-20b"
REASONING_EFFORT = "low"
print("✅ 環境：", "Colab" if IN_COLAB else "本地端", "｜模型：", MODEL)

✅ 環境： Colab ｜模型： openai/gpt-oss-20b


In [11]:
# 📋 規格書
'''
學號：411312155 姓名：張哲綸 卷別：B

## 1. 系統目的（一句話）
透過 AI 語意辨識讓學生能以自然語言預約、取消、查詢校園咖啡廳飲品庫存，並讓店長能隨時檢視今日銷售統計報表。

## 2. 使用者會怎麼說（至少 5 句）
1. 我要兩杯拿鐵
2. 我想買美式咖啡
3. 取消我的抹茶歐蕾
4. 查詢拿鐵庫存
5. 今天各品項賣得怎樣

## 3. 資料表（2 張）
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| menu | item_name | TEXT | 品項名稱（主鍵） |
| menu | price | INTEGER | 單價 |
| menu | stock | INTEGER | 今日剩餘庫存 |
| orders | order_id | INTEGER | 訂單編號（主鍵、自動遞增） |
| orders | student_name | TEXT | 訂購學生姓名 |
| orders | item_name | TEXT | 訂購品項 |
| orders | quantity | INTEGER | 訂購數量 |
| orders | status | TEXT | 訂單狀態（正常 / 已取消） |

初始資料：
- 美式咖啡：單價 50 元，庫存 5 杯
- 拿鐵：單價 65 元，庫存 4 杯
- 抹茶歐蕾：單價 70 元，庫存 3 杯

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,品項,數量`
- 動作只能是：訂購、取消、查詢、統計查詢、未知
- 品項只能是：美式咖啡、拿鐵、抹茶歐蕾、*
- 數量的規則：萃取數字，沒說時預設為 1
- 無法判斷時輸出：`未知,*,1`
- AI 不可以做的事：不可自行判斷庫存是否足夠、不可直接修改資料庫、不可回答與營業無關的內容。

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 訂購 | 庫存足夠且每人今日未超過 3 杯 | 顯示預約成功與總金額 | menu 庫存扣減，orders 新增一筆記錄 |
| 取消 | 查有該學生最新一筆有效訂購 | 顯示取消成功 | orders 狀態改為已取消，menu 庫存釋出 |
| 查詢 | 品項存在 | 顯示該品項單價與剩餘庫存 | 無 |
| 統計查詢 | 無條件 | 列出各品項單價、庫存、已售杯數、營收 | 無 (使用 LEFT JOIN 與 GROUP BY 查詢) |

## 6. 例外情況（至少 3 個）
1. 訂購時庫存不足，拒絕訂購並提示剩餘杯數。
2. 訂購數量不在 1~3 杯之間，或是單人當日累計數量超過 3 杯時拒絕訂購。
3. 取消訂購時查無該學生的有效訂單，提示無此訂單。

## 7. 測試清單
7-1 AI 解讀測試（8～10 句）與 7-2 流程測試（8～10 步）已於第 5 格與第 7 格程式中完整自動化測試呈現。
'''

'\n學號：411312155 姓名：張哲綸 卷別：B\n\n## 1. 系統目的（一句話）\n透過 AI 語意辨識讓學生能以自然語言預約、取消、查詢校園咖啡廳飲品庫存，並讓店長能隨時檢視今日銷售統計報表。\n\n## 2. 使用者會怎麼說（至少 5 句）\n1. 我要兩杯拿鐵\n2. 我想買美式咖啡\n3. 取消我的抹茶歐蕾\n4. 查詢拿鐵庫存\n5. 今天各品項賣得怎樣\n\n## 3. 資料表（2 張）\n| 表名 | 欄位 | 型態 | 說明 |\n|---|---|---|---|\n| menu | item_name | TEXT | 品項名稱（主鍵） |\n| menu | price | INTEGER | 單價 |\n| menu | stock | INTEGER | 今日剩餘庫存 |\n| orders | order_id | INTEGER | 訂單編號（主鍵、自動遞增） |\n| orders | student_name | TEXT | 訂購學生姓名 |\n| orders | item_name | TEXT | 訂購品項 |\n| orders | quantity | INTEGER | 訂購數量 |\n| orders | status | TEXT | 訂單狀態（正常 / 已取消） |\n\n初始資料：\n- 美式咖啡：單價 50 元，庫存 5 杯\n- 拿鐵：單價 65 元，庫存 4 杯\n- 抹茶歐蕾：單價 70 元，庫存 3 杯\n\n## 4. AI 的工作\n- 輸入：使用者的一句話\n- 輸出格式：`動作,品項,數量`\n- 動作只能是：訂購、取消、查詢、統計查詢、未知\n- 品項只能是：美式咖啡、拿鐵、抹茶歐蕾、*\n- 數量的規則：萃取數字，沒說時預設為 1\n- 無法判斷時輸出：`未知,*,1`\n- AI 不可以做的事：不可自行判斷庫存是否足夠、不可直接修改資料庫、不可回答與營業無關的內容。\n\n## 5. 功能與規則\n| 功能 | 條件 | 系統回應 | 資料庫變化 |\n|---|---|---|---|\n| 訂購 | 庫存足夠且每人今日未超過 3 杯 | 顯示預約成功與總金額 | menu 庫存扣減，orders 新增一筆記錄 |\n| 取消 | 查有該學生最新一筆有效訂購 | 顯示取消成

In [13]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================
import sqlite3

def init_db():
    conn = sqlite3.connect("cafe.db")
    cursor = conn.cursor()

    # 刪除舊資料表（確保每次執行都重建）
    cursor.execute("DROP TABLE IF EXISTS orders")
    cursor.execute("DROP TABLE IF EXISTS menu")

    # 建立菜單資料表 (menu)
    cursor.execute("""
        CREATE TABLE menu (
            item_name TEXT PRIMARY KEY,
            price INTEGER NOT NULL,
            stock INTEGER NOT NULL
        )
    """)

    # 建立訂單資料表 (orders)
    cursor.execute("""
        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY AUTOINCREMENT,
            student_name TEXT NOT NULL,
            item_name TEXT NOT NULL,
            quantity INTEGER NOT NULL,
            status TEXT NOT NULL,
            FOREIGN KEY (item_name) REFERENCES menu(item_name)
        )
    """)

    # 放入初始資料
    initial_menu = [
        ("美式咖啡", 50, 5),
        ("拿鐵", 65, 4),
        ("抹茶歐蕾", 70, 3)
    ]
    cursor.executemany("INSERT INTO menu (item_name, price, stock) VALUES (?, ?, ?)", initial_menu)

    conn.commit()
    conn.close()
    print("✅ 資料庫與初始資料建立完成！")

init_db()

✅ 資料庫與初始資料建立完成！


In [14]:
# ============================================================
# 第 3 格：顯示資料表
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================
import sqlite3
import pandas as pd
from IPython.display import display

def show_tables():
    conn = sqlite3.connect("cafe.db")
    print("--- 【 菜單 (menu) 】 ---")
    df_menu = pd.read_sql("SELECT * FROM menu", conn)
    display(df_menu)

    print("--- 【 訂單 (orders) 】 ---")
    df_orders = pd.read_sql("SELECT * FROM orders", conn)
    display(df_orders)
    conn.close()

# 呼叫一次以驗證初始化結果
show_tables()

--- 【 菜單 (menu) 】 ---


,item_name,price,stock
0,美式咖啡,50,5
1,拿鐵,65,4
2,抹茶歐蕾,70,3


--- 【 訂單 (orders) 】 ---


,order_id,student_name,item_name,quantity,status


In [15]:
# ============================================================
# 第 4 格：AI 解讀函式
# 提醒：回傳 (動作, 品項, 數量, 原始回覆)；不可直接相信 AI 的回覆
# ============================================================
import time

# 用字典暫存已辨識過的一句話，避免重複呼叫 AI
ai_cache = {}

def parse_with_ai(user_input):
    if user_input in ai_cache:
        return ai_cache[user_input]

    # 允許清單
    valid_actions = ["訂購", "取消", "查詢", "統計查詢"]
    valid_items = ["美式咖啡", "拿鐵", "抹茶歐蕾"]

    prompt = f"""
你是一個校園咖啡廳的語意解析助理。請分析學生的這句話：「{user_input}」
請嚴格依照以下格式輸出 3 個欄位，中間用逗號隔開：
動作,品項,數量

規則：
1. 動作只能從這幾個裡面選：訂購、取消、查詢、統計查詢。如果無法判斷或是在問營業/其他無關的事，動作請填「未知」。
2. 品項只能從這幾個裡面選：美式咖啡、拿鐵、抹茶歐蕾。如果沒有提到具體品項，請填「*」。
3. 數量請萃取阿拉伯數字（例如 1, 2, 3）。如果沒提到數量，預設為 1。
4. 千萬不要輸出其他多餘的字或解釋，只要輸出「動作,品項,數量」即可。
"""

    raw_response = ""
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                reasoning_effort=REASONING_EFFORT
            )
            raw_response = response.choices[0].message.content.strip()
            time.sleep(2)
            break
        except Exception as e:
            if "429" in str(e) and attempt < 2:
                time.sleep(30)
            else:
                raw_response = "未知,*,1"

    # 後處理與防呆：清理 AI 回覆
    parts = raw_response.split(",")
    action = parts[0].strip() if len(parts) > 0 else "未知"
    item = parts[1].strip() if len(parts) > 1 else "*"
    qty_str = parts[2].strip() if len(parts) > 2 else "1"

    # 檢查動作是否合法
    if action not in valid_actions:
        action = "未知"

    # 檢查品項是否合法
    if item not in valid_items:
        item = "*"

    # 用正規表示式抓出數量數字
    qty_match = re.search(r'\d+', qty_str)
    quantity = int(qty_match.group()) if qty_match else 1

    result = (action, item, quantity, raw_response)
    ai_cache[user_input] = result
    return result

# 簡單測試
print("解讀測試結果：", parse_with_ai("我要一杯拿鐵"))

解讀測試結果： ('訂購', '拿鐵', 1, '訂購,拿鐵,1')


In [16]:
# ============================================================
# 第 5 格：AI 解讀測試
# 提醒：8～10 句，要與規格書 7-1 一致
# ============================================================
test_cases = [
    ("我要兩杯拿鐵", "訂購", "拿鐵", 2),
    ("我想買美式咖啡", "訂購", "美式咖啡", 1),
    ("取消我的抹茶歐蕾", "取消", "抹茶歐蕾", 1),
    ("查詢拿鐵庫存", "查詢", "拿鐵", 1),
    ("今天各品項賣得怎樣", "統計查詢", "*", 1),
    ("我想訂 3 杯抹茶歐蕾", "訂購", "抹茶歐蕾", 3),
    ("幫我取消美式咖啡", "取消", "美式咖啡", 1),
    ("查一下美式咖啡還有多少", "查詢", "美式咖啡", 1),
    ("請問今天營業時間幾點到幾點", "未知", "*", 1)
]

passed_count = 0
print(f"總共進行 {len(test_cases)} 句 AI 解讀測試：\n" + "-"*50)

for idx, (text, exp_act, exp_item, exp_qty) in enumerate(test_cases, 1):
    action, item, qty, raw = parse_with_ai(text)
    is_pass = (action == exp_act and (exp_item == "*" or item == exp_item) and qty == exp_qty)
    if is_pass:
        passed_count += 1
    print(f"測試 {idx}: 「{text}」")
    print(f"  -> 預期: 動作={exp_act}, 品項={exp_item}, 數量={exp_qty}")
    print(f"  -> 實際: 動作={action}, 品項={item}, 數量={qty} (原始AI回應: {raw})")
    print(f"  -> 結果: {'✅ 通過' if is_pass else '❌ 失敗'}\n")

print(f"🎯 測試結算：總共通過 {passed_count} / {len(test_cases)} 句")

總共進行 9 句 AI 解讀測試：
--------------------------------------------------
測試 1: 「我要兩杯拿鐵」
  -> 預期: 動作=訂購, 品項=拿鐵, 數量=2
  -> 實際: 動作=訂購, 品項=拿鐵, 數量=2 (原始AI回應: 訂購,拿鐵,2)
  -> 結果: ✅ 通過

測試 2: 「我想買美式咖啡」
  -> 預期: 動作=訂購, 品項=美式咖啡, 數量=1
  -> 實際: 動作=訂購, 品項=美式咖啡, 數量=1 (原始AI回應: 訂購,美式咖啡,1)
  -> 結果: ✅ 通過

測試 3: 「取消我的抹茶歐蕾」
  -> 預期: 動作=取消, 品項=抹茶歐蕾, 數量=1
  -> 實際: 動作=取消, 品項=抹茶歐蕾, 數量=1 (原始AI回應: 取消,抹茶歐蕾,1)
  -> 結果: ✅ 通過

測試 4: 「查詢拿鐵庫存」
  -> 預期: 動作=查詢, 品項=拿鐵, 數量=1
  -> 實際: 動作=查詢, 品項=拿鐵, 數量=1 (原始AI回應: 查詢,拿鐵,1)
  -> 結果: ✅ 通過

測試 5: 「今天各品項賣得怎樣」
  -> 預期: 動作=統計查詢, 品項=*, 數量=1
  -> 實際: 動作=統計查詢, 品項=*, 數量=1 (原始AI回應: 統計查詢,*,1)
  -> 結果: ✅ 通過

測試 6: 「我想訂 3 杯抹茶歐蕾」
  -> 預期: 動作=訂購, 品項=抹茶歐蕾, 數量=3
  -> 實際: 動作=訂購, 品項=抹茶歐蕾, 數量=3 (原始AI回應: 訂購,抹茶歐蕾,3)
  -> 結果: ✅ 通過

測試 7: 「幫我取消美式咖啡」
  -> 預期: 動作=取消, 品項=美式咖啡, 數量=1
  -> 實際: 動作=取消, 品項=美式咖啡, 數量=1 (原始AI回應: 取消,美式咖啡,1)
  -> 結果: ✅ 通過

測試 8: 「查一下美式咖啡還有多少」
  -> 預期: 動作=查詢, 品項=美式咖啡, 數量=1
  -> 實際: 動作=查詢, 品項=美式咖啡, 數量=1 (原始AI回應: 查詢,美式咖啡,1)
  -> 結果: ✅ 通過

測試 9: 「請問今天營業時間幾點到幾點」
  -> 預期: 動作=未知, 品項=*, 數量=1

In [17]:
# ============================================================
# 第 6 格：功能函式
# 提醒：fetchone() 後先檢查 None；用 try/finally 關閉連線；統計用 JOIN＋GROUP BY
# ============================================================
import sqlite3

def process_order(student_name, item_name, quantity):
    if not student_name or student_name.strip() == "":
        return "❌ 訂購失敗：請輸入您的姓名！"
    if quantity < 1 or quantity > 3:
        return f"❌ 訂購失敗：每筆訂單數量限制為 1～3 杯（您輸入了 {quantity} 杯）。"

    conn = sqlite3.connect("cafe.db")
    cursor = conn.cursor()
    try:
        # 1. 檢查個人今日總杯數是否超過 3 杯上限（已取消的不計）
        cursor.execute("""
            SELECT SUM(quantity) FROM orders
            WHERE student_name = ? AND status = '正常'
        """, (student_name,))
        row = cursor.fetchone()
        current_total = row[0] if row and row[0] is not None else 0

        if current_total + quantity > 3:
            return f"❌ 訂購失敗 {student_name}：您今天已訂購 {current_total} 杯，加上這次會超過每人 3 杯的上限！"

        # 2. 檢查品項庫存與單價
        cursor.execute("SELECT price, stock FROM menu WHERE item_name = ?", (item_name,))
        menu_row = cursor.fetchone()
        if not menu_row:
            return f"❌ 訂購失敗：找不到品項「{item_name}」。目前品項有：美式咖啡、拿鐵、抹茶歐蕾。"

        price, stock = menu_row[0], menu_row[1]
        if stock < quantity:
            return f"❌ 訂購失敗：庫存不足！「{item_name}」目前僅剩 {stock} 杯。"

        # 3. 扣除庫存並寫入訂單
        new_stock = stock - quantity
        cursor.execute("UPDATE menu SET stock = ? WHERE item_name = ?", (new_stock, item_name))
        cursor.execute("""
            INSERT INTO orders (student_name, item_name, quantity, status)
            VALUES (?, ?, ?, '正常')
        """, (student_name, item_name, quantity))

        conn.commit()
        total_price = price * quantity
        return f"✅ 訂購成功！{student_name} 預約了 {quantity} 杯 {item_name}，總金額 {total_price} 元。目前該品項剩餘庫存：{new_stock} 杯。"
    finally:
        conn.close()

def process_cancel(student_name, item_name):
    if not student_name or student_name.strip() == "":
        return "❌ 取消失敗：請輸入您的姓名！"

    conn = sqlite3.connect("cafe.db")
    cursor = conn.cursor()
    try:
        # 尋找該學生該品項最新一筆「正常」狀態的訂單
        cursor.execute("""
            SELECT order_id, quantity FROM orders
            WHERE student_name = ? AND item_name = ? AND status = '正常'
            ORDER BY order_id DESC LIMIT 1
        """, (student_name, item_name))
        row = cursor.fetchone()
        if not row:
            return f"❌ 取消失敗：找不到 {student_name} 預約「{item_name}」的有效訂單。"

        order_id, qty = row[0], row[1]

        # 更新訂單狀態為「已取消」
        cursor.execute("UPDATE orders SET status = '已取消' WHERE order_id = ?", (order_id,))

        # 釋出庫存
        cursor.execute("UPDATE menu SET stock = stock + ? WHERE item_name = ?", (qty, item_name))
        conn.commit()
        return f"✅ 取消成功：已幫 {student_name} 取消編號 #{order_id} 的 {qty} 杯 {item_name} 訂單，並已釋出庫存。"
    finally:
        conn.close()

def process_query(item_name):
    conn = sqlite3.connect("cafe.db")
    cursor = conn.cursor()
    try:
        cursor.execute("SELECT stock, price FROM menu WHERE item_name = ?", (item_name,))
        row = cursor.fetchone()
        if not row:
            return f"❌ 查詢失敗：找不到品項「{item_name}」。"
        return f"🔍 查詢結果：「{item_name}」單價 {row[1]} 元，目前剩餘庫存：{row[0]} 杯。"
    finally:
        conn.close()

def process_statistics():
    conn = sqlite3.connect("cafe.db")
    cursor = conn.cursor()
    try:
        # 使用 JOIN 與 GROUP BY，確保沒有人訂的品項也必須列出 (LEFT JOIN)
        query = """
            SELECT
                m.item_name AS 品項,
                m.price AS 單價,
                m.stock AS 目前庫存,
                COALESCE(SUM(CASE WHEN o.status = '正常' THEN o.quantity ELSE 0 END), 0) AS 已售杯數,
                COALESCE(SUM(CASE WHEN o.status = '正常' THEN o.quantity * m.price ELSE 0 END), 0) AS 營收
            FROM menu m
            LEFT JOIN orders o ON m.item_name = o.item_name
            GROUP BY m.item_name, m.price, m.stock
        """
        df_stat = pd.read_sql(query, conn)
        return df_stat
    finally:
        conn.close()

print("✅ 功能函式載入完成！")

✅ 功能函式載入完成！


In [18]:
# ============================================================
# 第 7 格：主流程與流程測試
# 提醒：流程測試 8～10 步；最後呼叫 show_tables()
# ============================================================
import sqlite3

# 重建乾淨的資料庫來跑流程測試
init_db()

# 定義主控入口函式
def handle_user_request(student_name, user_input):
    action, item, qty, raw = parse_with_ai(user_input)

    if action == "訂購":
        if item == "*":
            return "🤖 系統回應：請告訴我您想訂購哪一種飲品（美式咖啡、拿鐵、抹茶歐蕾）。"
        return process_order(student_name, item, qty)

    elif action == "取消":
        if item == "*":
            return "🤖 系統回應：請告訴我您想取消哪一項飲品。"
        return process_cancel(student_name, item)

    elif action == "查詢":
        if item == "*":
            # 如果沒指定品項，列出所有品項與可用清單
            return "🤖 系統回應：本咖啡廳提供：美式咖啡、拿鐵、抹茶歐蕾。請問您想查詢哪一個？"
        return process_query(item)

    elif action == "統計查詢":
        df_s = process_statistics()
        # 轉成好看的字串回傳給網頁或測試印出
        result_str = "📊 【今日銷售統計報表】\n"
        for idx, row in df_s.iterrows():
            result_str += f"- {row['品項']} | 單價: {row['單價']}元 | 庫存: {row['目前庫存']}杯 | 已售: {row['已售杯數']}杯 | 營收: {row['營收']}元\n"
        return result_str.strip()

    else:
        return f"🤖 系統回應：聽不太懂您的需求。本咖啡廳提供美式咖啡、拿鐵、抹茶歐蕾，您可以說「我要兩杯拿鐵」、「查詢拿鐵庫存」或「今天各品項賣得怎樣」。"

# 執行 8～10 步流程測試
flow_tests = [
    ("小明", "我要兩杯拿鐵"),
    ("小明", "查詢拿鐵庫存"),
    ("小華", "我要一杯美式咖啡"),
    ("小華", "我要兩杯抹茶歐蕾"),
    ("小華", "今天各品項賣得怎樣"),
    ("小明", "取消我的拿鐵"),
    ("小明", "查詢拿鐵庫存"),
    ("小美", "我要四杯拿鐵"), # 故意超過數量上限 3 杯
    ("小美", "我要一杯拿鐵")
]

print("--- 【 開始執行 8～10 步流程測試 】 ---")
for step_idx, (name, text) in enumerate(flow_tests, 1):
    res = handle_user_request(name, text)
    print(f"步驟 {step_idx} | 姓名: {name} | 輸入: 「{text}」")
    print(f"結果: {res}\n" + "-"*40)

print("--- 【 流程測試結束後的資料表狀態 】 ---")
show_tables()

✅ 資料庫與初始資料建立完成！
--- 【 開始執行 8～10 步流程測試 】 ---
步驟 1 | 姓名: 小明 | 輸入: 「我要兩杯拿鐵」
結果: ✅ 訂購成功！小明 預約了 2 杯 拿鐵，總金額 130 元。目前該品項剩餘庫存：2 杯。
----------------------------------------
步驟 2 | 姓名: 小明 | 輸入: 「查詢拿鐵庫存」
結果: 🔍 查詢結果：「拿鐵」單價 65 元，目前剩餘庫存：2 杯。
----------------------------------------
步驟 3 | 姓名: 小華 | 輸入: 「我要一杯美式咖啡」
結果: ✅ 訂購成功！小華 預約了 1 杯 美式咖啡，總金額 50 元。目前該品項剩餘庫存：4 杯。
----------------------------------------
步驟 4 | 姓名: 小華 | 輸入: 「我要兩杯抹茶歐蕾」
結果: ✅ 訂購成功！小華 預約了 2 杯 抹茶歐蕾，總金額 140 元。目前該品項剩餘庫存：1 杯。
----------------------------------------
步驟 5 | 姓名: 小華 | 輸入: 「今天各品項賣得怎樣」
結果: 📊 【今日銷售統計報表】
- 抹茶歐蕾 | 單價: 70元 | 庫存: 1杯 | 已售: 2杯 | 營收: 140元
- 拿鐵 | 單價: 65元 | 庫存: 2杯 | 已售: 2杯 | 營收: 130元
- 美式咖啡 | 單價: 50元 | 庫存: 4杯 | 已售: 1杯 | 營收: 50元
----------------------------------------
步驟 6 | 姓名: 小明 | 輸入: 「取消我的拿鐵」
結果: ✅ 取消成功：已幫 小明 取消編號 #1 的 2 杯 拿鐵 訂單，並已釋出庫存。
----------------------------------------
步驟 7 | 姓名: 小明 | 輸入: 「查詢拿鐵庫存」
結果: 🔍 查詢結果：「拿鐵」單價 65 元，目前剩餘庫存：4 杯。
----------------------------------------
步驟 8 | 姓名: 小美 | 輸入: 「我要四杯拿鐵」
結果: ❌ 訂購失敗

,item_name,price,stock
0,美式咖啡,50,4
1,拿鐵,65,3
2,抹茶歐蕾,70,1


--- 【 訂單 (orders) 】 ---


,order_id,student_name,item_name,quantity,status
0,1,小明,拿鐵,2,已取消
1,2,小華,美式咖啡,1,正常
2,3,小華,抹茶歐蕾,2,正常
3,4,小美,拿鐵,1,正常


In [19]:
# ============================================================
# 第 8 格：Gradio 網頁
# 提醒：if IN_COLAB: demo.launch(share=True)
# ============================================================
import gradio as gr

def gradio_interface(student_name, user_input):
    if not student_name or not student_name.strip():
        return "⚠️ 請先在上方輸入您的「姓名」，才能進行預約或取消！"
    if not user_input or not user_input.strip():
        return "⚠️ 請輸入您想說的話（例如：我要一杯拿鐵、今天各品項賣得怎樣）。"

    return handle_user_request(student_name.strip(), user_input.strip())

# 建立 Gradio Blocks 介面
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ☕ 校園咖啡廳智慧預約與統計系統")
    gr.Markdown("歡迎使用！請在下方輸入您的姓名以及想說的一句話（支援：預約飲品、取消、查詢庫存、銷售統計）。")

    with gr.Row():
        name_input = gr.Textbox(label="您的姓名", placeholder="例如：張哲綸")

    with gr.Row():
        text_input = gr.Textbox(label="您的需求（一句話）", placeholder="例如：我要兩杯拿鐵，或是今天各品項賣得怎樣？")

    submit_btn = gr.Button("送出請求", variant="primary")
    output_box = gr.Textbox(label="系統回應", lines=5)

    submit_btn.click(
        fn=gradio_interface,
        inputs=[name_input, text_input],
        outputs=output_box
    )

# 啟動網頁 (支援 Colab share=True)
if __name__ == "__main__":
    if IN_COLAB:
        demo.launch(share=True)
    else:
        demo.launch()

/tmp/ipykernel_564/473444627.py:16: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5facf982684cb4962d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
